In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
from matplotlib.backends.backend_pdf import PdfPages
from collections import defaultdict

pd.options.display.max_rows=2000
pd.options.display.max_columns=70

plot_dir = "/Users/eric/repos/aud/plots"

remake_plots=False

In [ ]:
# Data importing stage

# Start with latest (?) cleaned data
cleaned_ema_path = "/Volumes/private/studydata/risk/data_processed/ema/ema.csv"
hourly_labels_path = "/Volumes/private/studydata/risk/data_processed/ema/labels_1hour.csv"
daily_labels_path = "/Volumes/private/studydata/risk/data_processed/ema/labels_1day.csv"
weekly_labels_path = "/Volumes/private/studydata/risk/data_processed/ema/labels_1week.csv"
study_dates_path = "/Volumes/private/studydata/risk/data_processed/ema/study_dates.csv" 

# Data imports
# Ecological momentary assessments
ema_raw = pd.read_csv(cleaned_ema_path,parse_dates=['dttm_obs'],date_parser=lambda x: pd.to_datetime(x, format="%Y-%m-%dT%H:%M:%S",utc=True).tz_convert('US/Central'))
# Data labels
hourly_labels_raw = pd.read_csv(hourly_labels_path,parse_dates=['dttm_label'],date_parser=lambda x: pd.to_datetime(x, format="%Y-%m-%dT%H:%M:%S",utc=True).tz_convert('US/Central'))
daily_labels_raw = pd.read_csv(daily_labels_path,parse_dates=['dttm_label'],date_parser=lambda x: pd.to_datetime(x, format="%Y-%m-%dT%H:%M:%S",utc=True).tz_convert('US/Central'))
weekly_labels_raw = pd.read_csv(weekly_labels_path,parse_dates=['dttm_label'],date_parser=lambda x: pd.to_datetime(x, format="%Y-%m-%dT%H:%M:%S",utc=True).tz_convert('US/Central'))
# Background information about the study dates for each individual
study_dates_raw = pd.read_csv(study_dates_path,parse_dates=['study_start','data_start','study_end','ema_end'],date_parser=lambda x: pd.to_datetime(x, format="%Y-%m-%dT%H:%M:%S",utc=True).tz_convert('US/Central'))

In [ ]:
# Miscellaneous helper functions

# Helper function for processing rising edges on lapses that span multiple continuous measurements
def lapse_bool_helper(row):
    # If it isn't a nan, just move on
    if not(np.isnan(row.lapse_bool_diff)):
        return row.lapse_bool_diff
    # NaN case
    else:
        # Detect a rising edge if the very first time period seen is a lapse
        if row.lapse_bool==1:
            return 1
        else:
            return 0

# Helper function for classifying individuals based on their lapse count
# Should refactor this to make it more flexible for different cutoffs
def lapse_id_classifier_helper(row,cutoffs):
    # If the person has no lapses, they are classified as a 0
    if row.lapse_count == 0:
        return 0
    # Cases for when they do lapse, currently divided into 4 quartiles
    else:
        if row.lapse_count <= cutoffs[0]:
            return 1
        elif row.lapse_count <= cutoffs[1]:
            return 2
        elif row.lapse_count <= cutoffs[2]:
            return 3
        else:
            return 4

# Checking for changes in subid within ema data
def ema_delta_helper(row):
    if row.id_change:
        return 0
    else:
        return (row.norm_date-row.norm_date_s1)*24

In [ ]:
# Sample format
# Labels
print("Sample label import (hourly)")
display(hourly_labels_raw.iloc[0:10])

print("Sample EMA import")
display(ema_raw.iloc[0:10])

In [ ]:
# Testing for missing time data (MR=missing rows)
# Set up dataframes
weekly_labels_mr = weekly_labels_raw.copy(deep=True)
daily_labels_mr = daily_labels_raw.copy(deep=True)
hourly_labels_mr = hourly_labels_raw.copy(deep=True)

# See overall shape of the dataframes (note the lengths)
print("Shapes of imported raw dataframes")
print("Weekly labels shape: {}".format(weekly_labels_mr.shape))
print("Daily labels shape: {}".format(daily_labels_mr.shape))
print("Hourly labels shape: {}".format(hourly_labels_mr.shape))
print()

# Add necessary information to identify >1hr gaps
# Define the reference time delta
hour_diff = pd.Timedelta(1,'hour')
# Weekly processing
weekly_labels_mr.sort_values(['subid','dttm_label'],inplace=True,ignore_index=True)
weekly_labels_mr['time_diff'] = weekly_labels_raw.dttm_label.diff()
weekly_mask = weekly_labels_raw.subid == weekly_labels_raw.subid.shift(1)
weekly_gap_count = weekly_labels_mr.loc[weekly_mask].query("time_diff != @hour_diff").shape[0]
# Daily processing
daily_labels_mr.sort_values(['subid','dttm_label'],inplace=True,ignore_index=True)
daily_labels_mr['time_diff'] = daily_labels_raw.dttm_label.diff()
daily_mask = daily_labels_raw.subid == daily_labels_raw.subid.shift(1)
daily_gap_count = daily_labels_mr.loc[daily_mask].query("time_diff != @hour_diff").shape[0]
# Hourly processing
hourly_labels_mr.sort_values(['subid','dttm_label'],inplace=True,ignore_index=True)
hourly_labels_mr['time_diff'] = hourly_labels_raw.dttm_label.diff()
hourly_mask = hourly_labels_raw.subid == hourly_labels_raw.subid.shift(1)
hourly_gap_count = hourly_labels_mr.loc[hourly_mask].query("time_diff != @hour_diff").shape[0]

# Present gap information
print("Example cases of missing data values from the weekly label dataset (line items where there is more than 1hr gap between rows)")
display(weekly_labels_mr.loc[weekly_mask].query("time_diff != @hour_diff").iloc[0:10])
print("Counts of gaps (does not account for gap length)")
print("Weekly labels: {}".format(weekly_gap_count))
print("Daily labels: {}".format(daily_gap_count))
print("Hourly labels: {}".format(hourly_gap_count))

print("Are these gaps here for a reason? Or should the construction of the label files be revisted?")
print("Also why is daily the longest file but weekly has fewer gaps?")

In [ ]:
# Begin analysis of label data

# Initial testing for hourly labels
# Join the relevant study date labels
hourly_labels_df = hourly_labels_raw.join(study_dates_raw.set_index('subid'),on='subid')

# Sort the information first by subid then datetime
hourly_labels_df.sort_values(['subid','dttm_label'],inplace=True,ignore_index=True)

# Calculate the time-delta between the start of data and datetime label (to normalize for people studied at different times)
hourly_labels_df['td_date']=hourly_labels_df.dttm_label.sub(hourly_labels_df.data_start)

# Convert the time-delta to an numeric value (in days)
hourly_labels_df['norm_date']=hourly_labels_df['td_date'].dt.days+hourly_labels_df['td_date'].dt.seconds/86400

# Get specifics of time of day/week
hourly_labels_df['day_of_week']=hourly_labels_df.apply(lambda row: row.dttm_label.day_name(),axis=1)
hourly_labels_df['day_of_week_num']=hourly_labels_df.apply(lambda row: row.dttm_label.weekday(),axis=1)

# Convert 'yes'/'no' to boolean for lapses
hourly_labels_df['lapse_bool'] = hourly_labels_df.apply(lambda row: 0 if row.lapse=='no' else 1,axis=1)

# Take the difference between the last row and the current row to find rising edges (i.e. the start of a multi-hour lapse)
hourly_labels_df['lapse_bool_diff']=hourly_labels_df.lapse_bool.diff()

# Make the rows where the subid changes NaN's (as if they were the beginning of the dataframe getting diff'd)
# First construct a mask where subid changes occur
mask = hourly_labels_df.subid != hourly_labels_df.subid.shift(1)
hourly_labels_df.loc[mask,'lapse_bool_diff']=np.nan

# Apply a correcting function that handles rising edge detection for the very first line of a subid
hourly_labels_df['lapse_bool_diff']=hourly_labels_df.apply(lapse_bool_helper,axis=1)

# Check the processed form of the dataframe
print("Processed label dataframe:")
hourly_labels_df.head()

In [ ]:
# See how many folks lapse as their first data points
print("Interesting to note that there is a reasonable number of people whose first hourly data corresponds to a lapse")
print("Is this just coincidence?")
print("Count: {}".format(hourly_labels_df.loc[mask].query("lapse_bool == 1").shape[0]))
display(hourly_labels_df.loc[mask].query("lapse_bool == 1"))

In [ ]:
# Take a look at a single subject
display(hourly_labels_df.query("subid==2 & lapse_bool==1"))

In [ ]:
# Create dataframe summarizing lapse behavior (each entry is a particular lapse)

# DEPRECATED - Discussed with John that the lapse is not a lapse state, but rather the start of a lapse (hence a sequential analysis over rows is unecessary)
# # Initialize a list for collecting entries
# lapse_summary_list = []
# # Assemble the subids included in analysis
# subids = hourly_labels_df.subid.unique()
# print("Including data from {} subjects".format(len(subids)))

# # Loop over individuals
# for id in subids:

#     # Make a dataframe for this subid only
#     temp_df = hourly_labels_df.query("subid==@id").copy()
#     # Initialize set of variables needed for processing in each loop
#     lapse_start = False # notes that a lapse has started
#     start_time = np.nan # stores the time when a lapse started
#     end_time = np.nan # final value for the lapse period
#     last_time = np.nan # time in the previous row

#     # Loop over the rows of the subid's dataframe
#     for i,row in temp_df.iterrows():
#         # If the diff value for this row is 1, this row must be a rising edge
#         if row.lapse_bool_diff==1:
#             # Print an error if the lapse_start is already true
#             if lapse_start == True:
#                 print("Processing error - lapse timer already started on rising edge")
#                 quit
#             # Flip start boolean and store start time
#             lapse_start=True
#             start_time=row.dttm_label
#             start_row = row.copy()
#         # If the diff value for this row is -1, this row must be a falling edge
#         if row.lapse_bool_diff==-1:
#             # Print an error if the lapse start is not already true
#             if lapse_start == False:
#                 print("Processing error - lapse timer not started on falling edge")
#                 quit
#             # Slightly more complex than needed due to data missingness
#             end_time = last_time + pd.Timedelta(1,'hour')
#             # Calculate delta and convert to an integer
#             delta = end_time-start_time
#             int_delta = delta.days + delta.seconds/86400
#             # Store all information as a dataframe and add it to the list
#             lapse_summary_list.append(pd.DataFrame({"subid":id,"lapse_start":start_time,"hour_of_day":start_time.hour,"lapse_duration":delta,"int_lapse_duration":int_delta,
#                                                     "norm_date":start_row.norm_date,"day_of_week":start_row.day_of_week,"day_of_week_num":start_row.day_of_week_num,
#                                                     "time_of_week":start_row.dttm_label.hour/24+start_row.day_of_week_num,"lapse":1},index=[0]))
#             # Reset all relevant variables
#             lapse_start = False
#             start_time = np.nan
#             end_time = np.nan
#         # Reset the last_time value to
#         last_time = row.dttm_label

# # Build the final dataframe from the list of dataframes (each of which describes one entry)
# lapse_summary_df = pd.concat(lapse_summary_list,ignore_index=True)
# print("Dataframe containing information on all lapses:")
# lapse_summary_df.head()

In [ ]:
# Path of the raw list of lapses
lapse_list_path = '/Volumes/private/studydata/risk/data_processed/shared/lapses.csv'
# Import and copy the dataframe
lapse_import_raw = pd.read_csv(lapse_list_path,parse_dates=['lapse_start','lapse_end'],date_parser=lambda x: pd.to_datetime(x, format="%Y-%m-%dT%H:%M:%S",utc=True).tz_convert('US/Central'))
lapse_summary_df = lapse_import_raw.drop(['ema_end'],axis=1).join(study_dates_raw.set_index('subid'),on='subid')
# Drop miscellaneous columns and get rid of 'exclude'==True rows
lapse_summary_df.drop(['lapse_start_date','lapse_start_time','lapse_end','lapse_end_time','lapse_end_date','response_id','lapse_cnt'],axis=1,inplace=True)
lapse_summary_df.drop(lapse_summary_df.loc[lapse_summary_df['exclude']==True].index, inplace=True)

# Re-create relevant columns from deprecated processing sequence above
#             lapse_summary_list.append(pd.DataFrame({"subid":id,"lapse_start":start_time,"hour_of_day":start_time.hour,"lapse_duration":delta,"int_lapse_duration":int_delta,
#                                                     "norm_date":start_row.norm_date,"day_of_week":start_row.day_of_week,"day_of_week_num":start_row.day_of_week_num,
#                                                     "time_of_week":start_row.dttm_label.hour/24+start_row.day_of_week_num,"lapse":1},index=[0]))
lapse_summary_df['td_date']=lapse_summary_df.lapse_start.sub(lapse_summary_df.data_start)
lapse_summary_df['norm_date']=lapse_summary_df['td_date'].dt.days+lapse_summary_df['td_date'].dt.seconds/86400
lapse_summary_df['hour_of_day']=lapse_summary_df.apply(lambda row: row.lapse_start.hour,axis=1)
lapse_summary_df['day_of_week']=lapse_summary_df.apply(lambda row: row.lapse_start.day_name(),axis=1)
lapse_summary_df['day_of_week_num']=lapse_summary_df.apply(lambda row: row.lapse_start.weekday(),axis=1)
lapse_summary_df['time_of_week']=lapse_summary_df.apply(lambda row: row.lapse_start.hour/24+row.day_of_week_num,axis=1)

# Final cleanup/drop
lapse_summary_df.rename(columns={'duration':'int_lapse_duration'},inplace=True)
lapse_summary_df.drop(['exclude','ema_1_6','study_start','data_start','study_end','ema_end'],inplace=True,axis=1)

# Take a peek
lapse_summary_df.head()

In [ ]:
# Create individual summary dataframe (each entry is an individual, all individuals accounted for)

# Count up the lapses
total_lapse_count = lapse_summary_df.shape[0]
# Initialize a list for collecting entries
individual_summary_list = []
# Gather up all the individuals in the lapse dataframe
lapsed_individuals = lapse_summary_df.subid.unique()

# Loop over individuals in the dataset
subids = hourly_labels_df.subid.unique()
for id in subids:
    # If there is record of a lapse, get count, avg duration, stdv of duration
    if id in lapsed_individuals:
        lapse_count = lapse_summary_df.groupby(['subid']).subid.count().loc[id]
        fractional_lapse_contribution = lapse_count/total_lapse_count
        lapse_duration_avg = lapse_summary_df.groupby(['subid']).int_lapse_duration.mean().loc[id]
        lapse_duration_std = lapse_summary_df.groupby(['subid']).int_lapse_duration.std().loc[id]
    # If the individual does not appear in the lapse dataframe, log 0s
    else:
        lapse_count = 0
        fractional_lapse_contribution=0
        lapse_duration_avg = 0
        lapse_duration_std = 0
    # Gather information about this individual and append to the dataframe list
    temp_df = pd.DataFrame({'subid':id,'lapse_count':lapse_count,'fractional_lapse_contribution':fractional_lapse_contribution,'lapse_duration_avg':lapse_duration_avg,'lapse_duration_std':lapse_duration_std},index=[0])
    individual_summary_list.append(temp_df)

# Build the final dataframe from the list of dataframes (each entry is an individual)
individual_summary_df = pd.concat(individual_summary_list,ignore_index=True)
print("Dataframe containing information about the lapse behavior of each individual:")
individual_summary_df.head()

In [ ]:
# Summarize the counts/proportions of lapses
print("Total subject count: {}".format(len(hourly_labels_df.subid.unique())))
print("Total count of lapses: {}".format(lapse_summary_df.shape[0]))

# Histogram of lapses by individual
fig,ax = plt.subplots(figsize=(10,6))
ax=sns.histplot(data=individual_summary_df,x='lapse_count',discrete=True)
ax.set_ylabel("Count of individuals")
ax.set_xlabel("Number of lapses during study")
ax.set_xticks(range(0,81,10))
ax.set_xlim([-5,85])
ax.set_title("Total lapse occurrences by individual (count)")
if remake_plots:
    fig.savefig(os.path.join(plot_dir,'lapse_count_hist.pdf'),bbox_inches='tight',facecolor='w')

# Histogram of lapses by individual
fig,ax = plt.subplots(figsize=(10,6))
ax=sns.histplot(data=individual_summary_df,x='lapse_count',discrete=True,stat='proportion')
ax.set_ylabel("Proportion of individuals")
ax.set_xlabel("Number of lapses during study")
ax.set_xticks(range(0,81,10))
ax.set_xlim([-5,85])
ax.set_ylim([0,0.5])
ax.set_title("Total lapse occurrences by individual (proportion)")
if remake_plots:
    fig.savefig(os.path.join(plot_dir,'lapse_prop_hist.pdf'),bbox_inches='tight',facecolor='w')

# ECDF of lapses
fig,ax = plt.subplots(figsize=(10,6))
ax=sns.ecdfplot(data=individual_summary_df,x='lapse_count')
ax.set_ylabel("Proportion of individuals")
ax.set_xlabel("Number of lapses during study")
ax.set_xticks(range(0,81,10))
ax.set_xlim([-5,85])
ax.set_ylim([-0.05,1.05])
ax.set_title("Total lapse occurrences by individual")
if remake_plots:
    fig.savefig(os.path.join(plot_dir,'lapse_count_ecdf.pdf'),bbox_inches='tight',facecolor='w')

print("Conclusions: Nearly half of individuals appear to have no lapses whatsoever, slow decay from 1-16")

# Proportional accounting of lapses from each lapse count
fig,ax = plt.subplots(figsize=(10,6))
ax=sns.barplot(data=individual_summary_df.groupby(['lapse_count'])['fractional_lapse_contribution'].sum().reset_index(),x='lapse_count',y='fractional_lapse_contribution')
ax.set_ylabel("Fraction of total lapses contributed")
ax.set_xlabel("Number of lapses during study")
#ax.set_xlim([0,18])
#ax.set_ylim([-0.05,1.05])
ax.set_title("Fraction of total lapses by individuals of each lapse count")
if remake_plots:
    fig.savefig(os.path.join(plot_dir,'lapse_fractional_bar.pdf'),bbox_inches='tight',facecolor='w')
print("Conclusions: Will need to consider some sort of metric for which predictions matter most (i.e. better to be more right overall or less wrong for any given individual?)")
# Prevent misc text output
None

In [ ]:
# Add histogram of lapse durations
# Expected to be incorrect until holes in the label data are fixed (noted 10/3)
fig,ax = plt.subplots(figsize=(10,6))
ax=sns.histplot(data=lapse_summary_df,x='int_lapse_duration',stat='proportion',discrete=True)
ax.set_ylabel("Proportion of individuals")
ax.set_xlabel("Lapse Duration (days)")
ax.set_xticks(range(18))
ax.set_title("Duration of lapses")
if remake_plots:
    fig.savefig(os.path.join(plot_dir,'lapse_duration_hist.pdf'),bbox_inches='tight',facecolor='w')
# Suppress output
None

In [ ]:
# Synthesize information about the number of lapses for further breakdowns in the subsequent plots
# Want to be able to categorize lapses as coming from individuals in upper 50% vs. lower 50%
lapse_dist = individual_summary_df.query("lapse_count!=0").lapse_count.to_numpy()

# Start by dividing people into quartiles
thresholds = np.quantile(lapse_dist,[0.25,0.5,0.75])
print("Lapse quartile thresholds (Q1, Q2, Q3): {}".format(thresholds))
print("Note: Non-lapsers assigned '0' label, lapsers assigned labels 1-4")
# Get the quartile of each individual
individual_summary_df['id_type']=individual_summary_df.apply(lambda row: lapse_id_classifier_helper(row,thresholds),axis=1)
print("Show that the id_type number has been added to each individual:")
display(individual_summary_df.iloc[0:10])

# Now add this information back into the lapse_summary_df dataframe (so each lapse can also be grouped by the type of individual)
lapse_summary_df = pd.merge(lapse_summary_df,individual_summary_df[['subid','id_type']],how='left',on='subid')
print("Show that the id_type number has been added to each lapse (tied back to the subid)")
display(lapse_summary_df.iloc[0:5])

In [ ]:
# Plot of WHEN lapses occur during study period
fig = plt.figure(layout='constrained',figsize=(20,10))
subfigs = fig.subfigures(1,2,wspace=0.1)
leftaxs = subfigs[0].subplots(3,1,gridspec_kw={'height_ratios':[0.25,0.5,0.25]})
rightaxs = subfigs[1].subplots(4,1)

# Plot the main histogram
ax=leftaxs[1]
sns.histplot(ax=ax,data=lapse_summary_df,x='norm_date',stat='proportion',binwidth=5,binrange=(0,90))
ax.set_ylabel("Proportion of lapses")
ax.set_xlabel("Day in study")
ax.set_xticks(range(0,100,10))
ax.set_title("Study days when lapses start in study (all subjects)")
leftaxs[0].set_visible(False)
leftaxs[2].set_visible(False)

# Plot the striped data
for i,type in enumerate([1,2,3,4]):
    ax = rightaxs[i]
    sns.histplot(ax=ax,data=lapse_summary_df.query("id_type==@type"),x='norm_date',stat='proportion',binwidth=5,binrange=(0,90))
    ax.set_ylabel("Proportion")
    ax.set_ylim([0,0.2])
    ax.set_title("Quartile {}".format(type))
    ax.set_xlabel("Day in study")
# Suppress output

if remake_plots:
    fig.savefig(os.path.join(plot_dir,'lapse_timing_hists.pdf'),bbox_inches='tight',facecolor='w')
None

print("Conclusions: Perhaps a slight downward trend in lapses over the study period, though we see variation within each quartile (upper quartile appears mostly steady)")

In [ ]:
# Plot of WHEN lapses occur during each week
fig = plt.figure(layout='constrained',figsize=(20,10))
subfigs = fig.subfigures(1,2,wspace=0.1)
leftaxs = subfigs[0].subplots(3,1,gridspec_kw={'height_ratios':[0.25,0.5,0.25]})
rightaxs = subfigs[1].subplots(4,1)

weekdays = ["Mon.","Tue.","Wed.","Thu.","Fri","Sat.","Sun."]
# Plot the main histogram
ax=leftaxs[1]
sns.histplot(ax=ax,data=lapse_summary_df,x='day_of_week_num',stat='proportion',discrete=True)
ax.set_ylabel("Proportion of lapses")
ax.set_xlabel("Day of week")
ax.set_xticks([0,1,2,3,4,5,6])
ax.set_xticklabels(weekdays)
ax.set_title("Day of week when lapses start in study (all subjects)")
leftaxs[0].set_visible(False)
leftaxs[2].set_visible(False)

# Plot the striped data
for i,type in enumerate([1,2,3,4]):
    ax = rightaxs[i]
    sns.histplot(ax=ax,data=lapse_summary_df.query("id_type==@type"),x='day_of_week_num',stat='proportion',discrete=True)
    ax.set_ylabel("Proportion")
    ax.set_ylim([0,0.3])
    ax.set_title("Quartile {}".format(type))
    ax.set_xlabel("Day of week")
    ax.set_xticks([0,1,2,3,4,5,6])
    ax.set_xticklabels(weekdays)
# Suppress output
None
if remake_plots:
    fig.savefig(os.path.join(plot_dir,'lapse_weekday_hists.pdf'),bbox_inches='tight',facecolor='w')

In [ ]:
# Plots of WHEN lapses occur during a given day
fig = plt.figure(layout='constrained',figsize=(20,10))
subfigs = fig.subfigures(1,2,wspace=0.1)
leftaxs = subfigs[0].subplots(3,1,gridspec_kw={'height_ratios':[0.25,0.5,0.25]})
rightaxs = subfigs[1].subplots(4,1)

# Plot the main histogram
ax=leftaxs[1]
sns.histplot(ax=ax,data=lapse_summary_df,x='hour_of_day',stat='proportion',discrete=True)
ax.set_ylabel("Proportion of lapses")
ax.set_xlabel("Hour of day")
ax.set_xticks(range(0,27,3))
ax.set_title("Hour of day when lapses start in study (all subjects)")
leftaxs[0].set_visible(False)
leftaxs[2].set_visible(False)

# Plot the striped data
for i,type in enumerate([1,2,3,4]):
    ax = rightaxs[i]
    sns.histplot(ax=ax,data=lapse_summary_df.query("id_type==@type"),x='hour_of_day',stat='proportion',discrete=True)
    ax.set_ylabel("Proportion")
    ax.set_ylim([0,0.25])
    ax.set_title("Quartile {}".format(type))
    ax.set_xlabel("Hour of day")
    ax.set_xticks(range(0,27,3))
# Suppress output
None
if remake_plots:
    fig.savefig(os.path.join(plot_dir,'lapse_hourofday_hists.pdf'),bbox_inches='tight',facecolor='w')

In [ ]:
# Plots of WHEN lapses occur during a given week (slightly richer, accounting for day and time)
fig = plt.figure(layout='constrained',figsize=(20,10))
subfigs = fig.subfigures(1,2,wspace=0.1)
leftaxs = subfigs[0].subplots(3,1,gridspec_kw={'height_ratios':[0.25,0.5,0.25]})
rightaxs = subfigs[1].subplots(4,1)

# Plot the main histogram
ax=leftaxs[1]
sns.histplot(ax=ax,data=lapse_summary_df,x='time_of_week',stat='proportion',binrange=[0,7],binwidth=0.125)
ax.set_ylabel("Proportion of lapses")
ax.set_xlabel("Day of week")
ax.set_xticks([0,1,2,3,4,5,6])
ax.set_xticklabels(weekdays)
ax.set_title("Time of week (day + hour) of lapse start in study (all subjects)")
leftaxs[0].set_visible(False)
leftaxs[2].set_visible(False)

# Plot the striped data
for i,type in enumerate([1,2,3,4]):
    ax = rightaxs[i]
    sns.histplot(ax=ax,data=lapse_summary_df.query("id_type==@type"),x='time_of_week',stat='proportion',binrange=[0,7],binwidth=0.125)
    ax.set_ylabel("Proportion")
    ax.set_ylim([0,0.12])
    ax.set_title("Quartile {}".format(type))
    ax.set_xlabel("Day of week")
    ax.set_xticks([0,1,2,3,4,5,6])
ax.set_xticklabels(weekdays)
# Suppress output
None
if remake_plots:
    fig.savefig(os.path.join(plot_dir,'lapse_timeofweek_hists.pdf'),bbox_inches='tight',facecolor='w')

In [ ]:
ema_raw.head()

In [ ]:
# Start taking a look at EMA responses

# Join the relevant study date labels
ema_df = ema_raw.join(study_dates_raw.set_index('subid'),on='subid')
# Drop the folks with no study date information
ema_df.dropna(axis=0,how='any',subset='study_start',inplace=True)

# Sort the information first by subid then datetime
ema_df.sort_values(['subid','dttm_obs'],inplace=True,ignore_index=True)

# Calculate the time-delta between the start of data and datetime label (to normalize for people studied at different times)
ema_df['td_date']=ema_df.dttm_obs.sub(ema_df.data_start)

# Convert the time-delta to an numeric value (in days)
ema_df['norm_date']=ema_df['td_date'].dt.days+ema_df['td_date'].dt.seconds/86400

# Get specifics of time of day/week
ema_df['day_of_week']=ema_df.apply(lambda row: row.dttm_obs.day_name(),axis=1)
ema_df['day_of_week_num']=ema_df.apply(lambda row: row.dttm_obs.weekday(),axis=1)

ema_df.head()

In [ ]:
# Noticed some weird rows
ema_df.query("norm_date>=100").head()

In [ ]:
# Drop these rows
drop_idx = ema_df.query("norm_date>=100").index.values.tolist()
ema_df.drop(drop_idx,inplace=True)
ema_df.query("norm_date>=100").head()
ema_df.reset_index()
None

In [ ]:
# Further processing of EMA data

# EMA gap investigation
# Want to see what the typical timing is in EMA responses

# New column for identifying subject changes
ema_df['subid_s1']=ema_df.subid.shift(1)
# Apply function to catch subject changes
ema_df['id_change']=ema_df.apply(lambda row: 0 if row.subid==row.subid_s1 else 1,axis=1)
# Get the shifted norm date (mostly deprecated)
ema_df['norm_date_s1']=ema_df.norm_date.shift(1)
# Take the floor of the date to get integer date numbers
ema_df["rounded_date"]=ema_df.apply(lambda row: np.floor(row.norm_date),axis=1)
# Get a numeric time of day
ema_df["time_of_day"]=ema_df.apply(lambda row: (row.norm_date-row.rounded_date)*24,axis=1)
# Take the difference between the norm date and the shifted norm date
ema_df['ema_delta']=ema_df.apply(ema_delta_helper,axis=1)
# Preallocate the EMA index (which EMA of the day) for each row
ema_df["ema_ind"]=np.nan

# Add the correct EMA index for each row
# Preallocate a list for gathering processed entries
ema_summary_list=[]
# Loop over all subjects
for id in subids:
    # Get the study days in the subject's experiment
    study_days = ema_df.query("subid==@id").rounded_date.unique()
    # Loop over the study days
    for day in study_days:
        # Get smaller dataframe dealing with just that subject and day
        day_df = ema_df.query("subid==@id and rounded_date==@day").copy(deep=True)
        ema_count = day_df.shape[0]
        
        # Check to see if multiple morning EMAs were submitted
        if (day_df.query("ema_type=='morning'").shape[0])>1:
            morning_inds = day_df.query("ema_type=='morning'").index.values.tolist()
            # ARBITRARY CONVENTION: only keep the last one
            for idx in morning_inds[0:len(morning_inds)-1]:
                day_df.drop(index=idx,inplace=True)
                ema_df.drop(index=idx,inplace=True)
        # Get indices of dataframe as a list
        indices = day_df.index.values.tolist()
        # Cleanest case, morning is the first recorded value
        if day_df.iloc[0].ema_type=='morning':
            # Assign EMA index starting at 0
            for i,idx in enumerate(indices):
                ema_df.loc[idx,'ema_ind']=i
        # Exception dases
        else:
            # Test for the case where there is no morning EMA
            morning = day_df.query("ema_type=='morning'").shape[0]
            # No morning EMAs, ARBITRARY CONVENTION: treat these as observations starting at index 1
            if morning==0:
                for i,idx in enumerate(indices):
                    ema_df.loc[idx,'ema_ind']=i+1
            # Morning EMA present, assume prior EMAs are from previous day
            else:
                prev_day = day - 1
                # Odd case when this exception occurs at the beginning of the subject's experiment
                if prev_day<0:
                    print("Missing entry on first day for {}".format(id))
                    pre_ind=4
                # Mid-study case
                else:
                    # Get information from previous day
                    prev_day_df = ema_df.query("subid==@id & rounded_date==@prev_day").copy(deep=True)
                    # If there's no information from the previous day, ARBITRARY CONVENTION: start indexing at 1
                    if prev_day_df.shape[0]==0:
                        pre_ind=1
                    # Otherwise start indexing from last day's EMA index +1
                    else:
                        prev_max_ind = prev_day_df.ema_ind.max()
                        pre_ind = prev_max_ind +1
                # Start iterating over the rows
                for i,idx in enumerate(indices):
                    # Get the current row, need to check for when the shift from later to morning EMA occurs
                    curr_row = day_df.loc[idx]
                    # If it's a later EMA, start labeling using the index starting point decided above
                    if curr_row.ema_type=='later':
                        ema_df.loc[idx,'ema_ind']=pre_ind
                        # Also shift the time so that it happens > 24 hrs (since it's a spillover from the previous day)
                        curr_time = ema_df.loc[idx,'time_of_day']
                        ema_df.loc[idx,'time_of_day']=curr_time+24
                        # Set the rounded date back a day
                        ema_df.loc[idx,'rounded_date']=day-1
                        pre_ind+=1
                    # This means we've found a morning EMA
                    else:
                        # Reset the indices vector so that labeling can happen as normal in the 'clean' case
                        morning_ind=idx
                        list_ind = indices.index(morning_ind)
                        indices=indices[list_ind:]
                        break
                # Now label items as normal (note that this indices vector is now reduced to only the elements from the morning EMA onward - the other EMAs will have been labeled by the previous loop)
                for i,idx in enumerate(indices):
                    ema_df.loc[idx,'ema_ind']=i

        # Add this entry to the dataframe list
        temp_df = pd.DataFrame({'subid':id,'day_in_study':day,'emas_completed':ema_count},index=[0])
        ema_summary_list.append(temp_df)

# Build the final dataframe from the list of dataframes (each entry is an individual)
ema_summary_df = pd.concat(ema_summary_list,ignore_index=True)

In [ ]:
# Check to see how many EMAs folks fill out
fig,ax = plt.subplots(figsize=(10,6))
ax=sns.histplot(data=ema_df['subid'].value_counts(),binwidth=10)
ax.set_ylabel("Count of individuals")
ax.set_xlabel("Number of EMAs completed during study")
#ax.set_yticks(range(0,30,2))
ax.set_title("Total EMAs completed by subject")
if remake_plots:
    fig.savefig(os.path.join(plot_dir,'ema_count_hist.pdf'),bbox_inches='tight',facecolor='w')
None
print("May be worthwhile to filter the inviduals below a certain threshold for a sanity check when training models")

fig,ax = plt.subplots(figsize=(10,6))
ax=sns.histplot(data=ema_df.query("ema_type=='morning'")['subid'].value_counts(),binwidth=10)
ax.set_ylabel("Count of individuals")
ax.set_xlabel("Number of Morning EMAs completed during study")
#ax.set_yticks(range(0,30,2))
ax.set_title("Morning EMAs completed by subject")
if remake_plots:
    fig.savefig(os.path.join(plot_dir,'morning_ema_count_hist.pdf'),bbox_inches='tight',facecolor='w')
None

In [ ]:
lapse_summary_df.query("subid==16")

In [ ]:
# Subject plotter script
def ema_plotter(id,df_in,lapse_summary,individual_summary):
    # Gather lapse times
    lapse_times = lapse_summary_df.query("subid==@id").norm_date.to_list()
    rolling_window = 4
    # Create figure
    fig = plt.figure(figsize=(25,20),layout='constrained')
    subfigs = fig.subfigures(nrows=2,ncols=1,wspace=0.1,height_ratios=[3,1.25])
    axs=subfigs[0].subplots(nrows=3,ncols=3)
    #fig,axs = plt.subplots(figsize=(20,10),nrows=3,ncols=3,constrained_layout=True)

    # Get individual information
    #df = ema_df.query("subid==@id").copy()
    df = df_in.query("subid==@id").copy()

    # Q2 - Since your last survey, how intense was your greatest urge to drink alcohol? (0:no urges - 12:strong)
    ax=axs[0,0]
    sns.scatterplot(ax=ax,data=df,x='norm_date',y='ema_2')
    sns.lineplot(ax=ax,x=df.norm_date,y=df.ema_2.rolling(rolling_window).mean(),c='g',drawstyle='steps-mid')
    ax.vlines(x=lapse_times, ymin=-2,ymax=15,color='r', ls='--', lw=1)
    ax.set_ylim([-0.5,12.5])
    ax.set_title("Q2 - Urge to drink")
    ax.set_xlabel("Date in study")
    ax.set_ylabel("Response")

    # Q3 - Since your last survey, did you encounter any risky situations? Rate it.
    ax=axs[0,1]
    sns.scatterplot(ax=ax,data=df,x='norm_date',y='ema_3')
    sns.lineplot(ax=ax,x=df.norm_date,y=df.ema_3.rolling(rolling_window).mean(),c='g',drawstyle='steps-mid')
    ax.vlines(x=lapse_times, ymin=-2,ymax=15,color='r', ls='--', lw=1)
    ax.set_ylim([-0.5,12.5])
    ax.set_title("Q3 - Intensity of risky situation")
    ax.set_xlabel("Date in study")
    ax.set_ylabel("Response")

    # Q4 - Since your last survey, has a hassle or stressful event occurred? Rate it.
    ax=axs[0,2]
    sns.scatterplot(ax=ax,data=df,x='norm_date',y='ema_4')
    sns.lineplot(ax=ax,x=df.norm_date,y=df.ema_4.rolling(rolling_window).mean(),c='g',drawstyle='steps-mid')
    ax.vlines(x=lapse_times, ymin=-2,ymax=15,color='r', ls='--', lw=1)
    ax.set_ylim([-0.5,12.5])
    ax.set_title("Q4 - Intensity of hassle/stressful situation")
    ax.set_xlabel("Date in study")
    ax.set_ylabel("Response")

    # Q5 - Since your last surevy has a pleasant or positive event occurred? Rate it.
    ax=axs[1,0]
    sns.scatterplot(ax=ax,data=df,x='norm_date',y='ema_5')
    sns.lineplot(ax=ax,x=df.norm_date,y=df.ema_5.rolling(rolling_window).mean(),c='g',drawstyle='steps-mid')
    ax.vlines(x=lapse_times, ymin=-2,ymax=15,color='r', ls='--', lw=1)
    ax.set_ylim([-0.5,12.5])
    ax.set_title("Q5 - Intensity of pleasant/positive experience")
    ax.set_xlabel("Date in study")
    ax.set_ylabel("Response")
    fig.suptitle("Subject {}".format(id))

    # Q6 - How are you feeling right now? (1:unhappy - 11:happy)
    ax=axs[1,1]
    sns.scatterplot(ax=ax,data=df,x='norm_date',y='ema_6')
    sns.lineplot(ax=ax,x=df.norm_date,y=df.ema_6.rolling(rolling_window).mean(),c='g',drawstyle='steps-mid')
    ax.vlines(x=lapse_times, ymin=-2,ymax=15,color='r', ls='--', lw=1)
    ax.set_ylim([-0.5,12.5])
    ax.set_title("Q6 - Unhappy/Happy")
    ax.set_xlabel("Date in study")
    ax.set_ylabel("Response")

    # Q7 - How are you feeling right now? (1:calm - 11:aroused)
    ax=axs[1,2]
    sns.scatterplot(ax=ax,data=df,x='norm_date',y='ema_7')
    sns.lineplot(ax=ax,x=df.norm_date,y=df.ema_7.rolling(rolling_window).mean(),c='g',drawstyle='steps-mid')
    ax.vlines(x=lapse_times, ymin=-2,ymax=15,color='r', ls='--', lw=1)
    ax.set_ylim([-0.5,12.5])
    ax.set_title("Q7 - Calm/Aroused")
    ax.set_xlabel("Date in study")
    ax.set_ylabel("Response")

    # Q8 - How likely to encounter risky situations within the next week?
    ax=axs[2,0]
    sns.scatterplot(ax=ax,data=df,x='norm_date',y='ema_8')
    sns.lineplot(ax=ax,x=df.query("ema_type=='morning'").norm_date,y=df.query("ema_type=='morning'").ema_8.rolling(rolling_window).mean(),c='g',drawstyle='steps-mid')
    ax.vlines(x=lapse_times, ymin=-2,ymax=15,color='r', ls='--', lw=1)
    ax.set_ylim([-0.5,12.5])
    ax.set_title("Q8 - How likely to encounter risky situations")
    ax.set_xlabel("Date in study")
    ax.set_ylabel("Response")

    # Q9 - How likely to encounter stressful event within the next week?
    ax=axs[2,1]
    sns.scatterplot(ax=ax,data=df,x='norm_date',y='ema_9')
    sns.lineplot(ax=ax,x=df.query("ema_type=='morning'").norm_date,y=df.query("ema_type=='morning'").ema_9.rolling(rolling_window).mean(),c='g',drawstyle='steps-mid')
    ax.vlines(x=lapse_times, ymin=-2,ymax=15,color='r', ls='--', lw=1)
    ax.set_ylim([-0.5,12.5])
    ax.set_title("Q9 - How likely to encounter stressful situations")
    ax.set_xlabel("Date in study")
    ax.set_ylabel("Response")

    # Q10 - How likely to drink alcohol within the next week?
    ax=axs[2,2]
    sns.scatterplot(ax=ax,data=df,x='norm_date',y='ema_10')
    sns.lineplot(ax=ax,x=df.query("ema_type=='morning'").norm_date,y=df.query("ema_type=='morning'").ema_10.rolling(rolling_window).mean(),c='g',drawstyle='steps-mid')
    ax.vlines(x=lapse_times, ymin=-2,ymax=15,color='r', ls='--', lw=1)
    ax.set_ylim([-0.5,12.5])
    ax.set_title("Q10 - How likely to drink alcohol in next week?")
    ax.set_xlabel("Date in study")
    ax.set_ylabel("Response")

    ax=subfigs[1].subplots()
    # Plot EMA 1
    sns.lineplot(ax=ax,data=df.query("ema_ind==0"),x='rounded_date',y='time_of_day',c='k',marker='o',linestyle='--')
    sns.lineplot(ax=ax,data=df.query("ema_ind==1"),x='rounded_date',y='time_of_day',c='y',marker='o',linestyle='--')
    sns.lineplot(ax=ax,data=df.query("ema_ind==2"),x='rounded_date',y='time_of_day',c='r',marker='o',linestyle='--')
    sns.lineplot(ax=ax,data=df.query("ema_ind==3"),x='rounded_date',y='time_of_day',c='b',marker='o',linestyle='--')
    sns.lineplot(ax=ax,data=df.query("ema_ind==4"),x='rounded_date',y='time_of_day',c='c',marker='o',linestyle='--')
    # Plot EMA 
    #sns.lineplot(ax=ax,x=df.norm_date,y=df.ema_2.rolling(rolling_window).mean(),c='g',drawstyle='steps-mid')
    ax.vlines(x=lapse_times, ymin=-2,ymax=30,color='r', ls='--', lw=1)
    ax.set_ylim([-0.5,35.5])
    ax.set_title("EMA Response Times")
    ax.set_ylabel("Hour of day")
    ax.set_xlabel("Day in study")

    # Add subject title
    lapse_count = len(lapse_times)
    id_type=individual_summary_df.query("subid==@id").id_type.values[0]
    fig.suptitle("Subject {} - {} total lapses, Q{}, Rolling window = {}".format(id,lapse_count,id_type,rolling_window))
    # Comment for plot functionality testing
    plt.close(fig)
    return fig,axs

In [ ]:
# Test plotting functionality (comment out the fig.close above)
#fig,axs = ema_plotter(2)

In [ ]:
# Create a pdfpages store of each individuals EMA responses
subids = ema_df.subid.unique()
# NOTE: The data in the labels has fewer subids than the EMA data (need to clarify with John)
if remake_plots:
    complete_store=PdfPages(os.path.join(plot_dir,"subject_emas.pdf"))
    for id in subids:
        fig,axs = ema_plotter(id,ema_df,lapse_summary_df,individual_summary_df)
        complete_store.savefig(fig,bbox_inches='tight',facecolor='w')
    complete_store.close()

# Beginning of regression testing

In [ ]:
#Load packages
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split,StratifiedKFold,RepeatedStratifiedKFold,GridSearchCV, RandomizedSearchCV
from sklearn.metrics import roc_curve, precision_recall_curve, PrecisionRecallDisplay, RocCurveDisplay, roc_auc_score, precision_score, recall_score, confusion_matrix,average_precision_score
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from scipy.stats import randint

In [ ]:
study_dates_raw.head()

In [ ]:
# Create a day-level dataset (day is labeled 1 if a lapse starts on that day, 0 otherwise)
label_list=[]

# Shift corresponds to a day that starts and ends at 4AM
day_shift = 0.16
# Get the subids from the labels dataset that was originally imported
subids=hourly_labels_df.subid.unique()
# Loop over the subjects
for id in subids:
    # Identify the day of the week for day 0 for this particular patient
    data_start_day_of_week=study_dates_raw.query("subid==@id").data_start.iloc[0].weekday()
    # Stick to standard 90 day study period
    for day in range(90):
        # Calculate the day of the week based on the day of the week at the beginning of the study
        day_of_week_num=(data_start_day_of_week+day)%7

        # Start time is either 0 (for day zero) or 4AM
        if day!=0:
            start=day+day_shift
        else:
            start=day
        # End of day
        end = day+day_shift+1

        # Gather the entries in the particular (start,end) window
        lapse_count = lapse_summary_df.query("subid==@id & @start<norm_date<@end").shape[0]
        lapse_bool = 1 if lapse_count>=1 else 0

        # Add this information to a dataframe and append to the concatenation list
        temp_df = pd.DataFrame({'subid':id,'day':day,'lapse_count_d0':lapse_count,'lapse_bool_d0':lapse_bool,'day_of_week_num':day_of_week_num},index=[0])
        label_list.append(temp_df)
        
# Create a dataframe from the list
day_labels = pd.concat(label_list,ignore_index=True)
day_labels = pd.merge(day_labels,individual_summary_df[['subid','id_type','lapse_count']],how='left',on='subid')
day_labels = pd.get_dummies(day_labels,columns=['day_of_week_num'],drop_first=True)
day_labels.head()

In [ ]:
# Create additional label columns for shifted days
day_shift = [1,2,3,4,5,6,7,8,9,10,11,12,13,14]
for delta in day_shift:
    col_name = 'lapse_bool_d'+str(delta)
    day_labels[col_name]=day_labels.groupby('subid')['lapse_bool_d0'].shift(-delta)

In [ ]:
# Create additional (feature) columns for whether there was a lapse in the last 1, 3, or 5 days
other_shift = [1,2,3,4,5]
for delta in other_shift:
    col_name = 'lapse_bool_minus'+str(delta)
    day_labels[col_name]=day_labels.groupby('subid')['lapse_bool_d0'].shift(delta)
    day_labels[col_name]=day_labels[col_name].fillna(0)

day_labels['minus_1']=day_labels.apply(lambda row: row.lapse_bool_minus1,axis=1)
day_labels['minus_3']=day_labels.apply(lambda row: row.lapse_bool_minus1 or row.lapse_bool_minus2 or row.lapse_bool_minus3,axis=1)
day_labels['minus_5']=day_labels.apply(lambda row: row.lapse_bool_minus1 or row.lapse_bool_minus2 or row.lapse_bool_minus3 or row.lapse_bool_minus4 or row.lapse_bool_minus5,axis=1)
day_labels.drop(labels=['lapse_bool_minus1','lapse_bool_minus2','lapse_bool_minus3','lapse_bool_minus4','lapse_bool_minus5'],axis=1,inplace=True)
day_labels.head()

In [ ]:
# Helper function for identifying if there is a lapse within a given week
def calculate_week_lapse(row,shift):
    col_labels = ['lapse_bool_d'+str(i) for i in range(shift,shift+7,1)]
    weekvals = row[col_labels].tolist()
    if np.isnan(weekvals).any():
        return np.nan
    else:
        return max(weekvals)
week_shift = [0,1,2,3,4,5,6,7]   
for shift in week_shift:
    col_name = 'lapse_bool_w'+str(shift)
    day_labels[col_name]=day_labels.apply(lambda row: calculate_week_lapse(row,shift),axis=1)

In [ ]:
day_labels.head()

In [ ]:
# Generate dataframe describing EMA compliance for use in subject selection
ema_compliance_list=[]
for id in day_labels.subid.unique():
    # Get current subjects EMAs
    sub_df = ema_df.query("subid==@id").copy()
    # Total EMAs
    total = sub_df.shape[0]
    # Morning/later EMAs
    morning = sub_df.query("ema_type=='morning'").shape[0]
    later = total-morning
    # Days with EMAs
    morning_day = sub_df.query("ema_type=='morning'").rounded_date.unique().shape[0]
    later_day = sub_df.query("ema_type=='later'").rounded_date.unique().shape[0]
    temp_df = pd.DataFrame({'subid':id,'total_ema':total,'morning_ema':morning,'later_ema':later,'days_w_morning':morning_day,'days_w_later':later_day},index=[0])
    ema_compliance_list.append(temp_df)
# Create a dataframe from the list
ema_compliance = pd.concat(ema_compliance_list,ignore_index=True)
ema_compliance.head()

In [ ]:
# Featurizer helper function
def make_feats(labels,emas,subids):
    # Create the framework from the label dataset (just adding the feature columns to the existing label dataframe)
    df = labels.copy(deep=True)
    # Create the relevant columns and initialize to NaNs
    df[['ema_2','ema_3','ema_4','ema_5','ema_6','ema_7','ema_8','ema_9','ema_10',
        'lrm_ema_2','lrm_ema_3','lrm_ema_4','lrm_ema_5','lrm_ema_6','lrm_ema_7','lrm_ema_8','lrm_ema_9','lrm_ema_10',
        'srm_ema_2','srm_ema_3','srm_ema_4','srm_ema_5','srm_ema_6','srm_ema_7','srm_ema_8','srm_ema_9','srm_ema_10']]=np.nan#,
        #'d_lrm_ema_2','d_lrm_ema_3','d_lrm_ema_4','d_lrm_ema_5','d_lrm_ema_6','d_lrm_ema_7','d_lrm_ema_8','d_lrm_ema_9','d_lrm_ema_10']]=np.nan

    # Loop over the provided subids
    for id in subids:
        # Create a dataframe for this subject and filter to their morning EMAs
        sub_df = emas.query("subid==@id & ema_type=='morning'").copy(deep=True)
        study_days = df.day.unique()
        # Loop over all days
        for day in study_days:
            # Further filter to days <= target day, with a descending sort (such that the first entry is the most recent)
            filt_df = sub_df.query("rounded_date<=@day").copy().sort_values(by='norm_date',ascending=False)
            # Get the indices of the filtered dataframe
            idxs = df.query("subid==@id & day==@day").index.tolist()
            # Sanity check for multiple morning EMAs for this day
            if len(idxs)>1:
                print("ERROR on participant {}, day {}".format(i,day))
                break
            else:
                day_idx=idxs[0]
            # Get the relevant EMA values (look for the most recent non-NaN entries)
            # Generate a dummy row in rare cases where there is no data
            if filt_df.dropna(subset=['ema_2','ema_3','ema_4','ema_5','ema_6','ema_7','ema_8','ema_9','ema_10']).shape[0]==0:
                temp=filt_df.reindex([0])
                recent_vals = temp.iloc[0]
            else:
                recent_vals=filt_df.dropna(subset=['ema_2','ema_3','ema_4','ema_5','ema_6','ema_7','ema_8','ema_9','ema_10']).iloc[0]
            # Put these values into the dataframe in the correct row
            raw_vals = recent_vals[['ema_2','ema_3','ema_4','ema_5','ema_6','ema_7','ema_8','ema_9','ema_10']].tolist()
            df.loc[day_idx,['ema_2','ema_3','ema_4','ema_5','ema_6','ema_7','ema_8','ema_9','ema_10']]=raw_vals

            # Short run mean (SRM) values
            len_df = filt_df.shape[0]
            end_idx = min(len_df,3)
            if end_idx==0:
                temp=filt_df.reindex([0])
                srm_vals = temp.iloc[0]
            else:
                srm_vals = filt_df.iloc[0:end_idx][['ema_2','ema_3','ema_4','ema_5','ema_6','ema_7','ema_8','ema_9','ema_10']].mean().tolist()
            df.loc[day_idx,['srm_ema_2','srm_ema_3','srm_ema_4','srm_ema_5','srm_ema_6','srm_ema_7','srm_ema_8','srm_ema_9','srm_ema_10']]=srm_vals
            # Long run mean (LRM) values
            lrm_vals = filt_df[['ema_2','ema_3','ema_4','ema_5','ema_6','ema_7','ema_8','ema_9','ema_10']].mean().tolist()
            df.loc[day_idx,['lrm_ema_2','lrm_ema_3','lrm_ema_4','lrm_ema_5','lrm_ema_6','lrm_ema_7','lrm_ema_8','lrm_ema_9','lrm_ema_10']]=lrm_vals

            # Delta long run mean values
            #df.loc[day_idx,['d_lrm_ema_2','d_lrm_ema_3','d_lrm_ema_4','d_lrm_ema_5','d_lrm_ema_6','d_lrm_ema_7','d_lrm_ema_8','d_lrm_ema_9','d_lrm_ema_10']]=np.array(raw_vals)-np.array(lrm_vals)
    return df

In [ ]:
# Make features, including a long-running arithmetic mean (from study start to present) and a short running arithmetic mean (last 3 morning EMA observations)
subids = ema_compliance.query("days_w_morning>=60").subid.unique()
feature_df = make_feats(day_labels.query("subid in @subids"),ema_df.query("subid in @subids"),subids)

In [ ]:
feature_df.head()

In [ ]:
day_list = [0]+day_shift

day_test_labels = ['lapse_count_d0']+['lapse_bool_d'+str(i) for i in day_list]
week_test_labels = ['lapse_bool_w'+str(i) for i in week_shift]
misc_labels = ['subid','day','id_type','lapse_count']
minus_labels = ['minus_1','minus_3','minus_5']
ema_labels = ['ema_2','ema_3','ema_4','ema_5','ema_6','ema_7','ema_8','ema_9','ema_10']
srm_labels = ['srm_ema_2','srm_ema_3','srm_ema_4','srm_ema_5','srm_ema_6','srm_ema_7','srm_ema_8','srm_ema_9','srm_ema_10']
lrm_labels = ['lrm_ema_2','lrm_ema_3','lrm_ema_4','lrm_ema_5','lrm_ema_6','lrm_ema_7','lrm_ema_8','lrm_ema_9','lrm_ema_10']
dow_labels = ['day_of_week_num_1','day_of_week_num_2','day_of_week_num_3','day_of_week_num_4','day_of_week_num_5','day_of_week_num_6']

print(feature_df.shape)
clean_feat = feature_df.dropna(subset=ema_labels+srm_labels+lrm_labels+dow_labels+minus_labels,axis=0)
# The current featurization uses the most recent available EMA when there is no EMA for a particular day
# The only days to be dropped are those in which there is no previous EMA to pull from (i.e., missing data at the beginning of the study)
only_na = feature_df[~feature_df.index.isin(clean_feat.index)]
print(clean_feat.shape)
clean_feat.head()
print(day_test_labels)
only_na.head()

In [ ]:
# Data imbalance check
no_lapse = day_labels.query("lapse_bool_d0 == 0").shape[0]
lapse = day_labels.shape[0]-no_lapse
print("Total entries: ",no_lapse+lapse)
print("Lapse entries: ",lapse)
print("Lapse entry fraction: {:.3f}".format(lapse/(no_lapse+lapse)))

In [ ]:
# Correlation check
corr = clean_feat.corr()
corr.style.background_gradient(cmap='coolwarm')

# Logistic Regression

In [ ]:
# Design a set of possible features
logreg_models= {
    # EMA data only, with two seemingly unimportant values dropped
    'limited_ema_only':{
        'drop':srm_labels+lrm_labels+dow_labels+['ema_6','ema_9']+minus_labels
    },
    # EMA responses only
    'ema_only':{
        'drop':srm_labels+lrm_labels+dow_labels+minus_labels
    },
    # EMA responses + day of week
    'ema':{
        'drop':srm_labels+lrm_labels+minus_labels
    },
    # EMA responses + day of week + long run mean values of EMA
    'ema_lrm':{
        'drop':srm_labels+minus_labels
    },
    # EMA responses + day of week + short run mean values of EMA
    'ema_srm':{
        'drop':lrm_labels+minus_labels
    },
    # EMA responses + day of week + short run mean + long run mean EMA values
    'ema_srm_lrm':{
        'drop':minus_labels
    },
    'ema_srm_lrm_recent_lapse':{
        'drop':[]
    }
}
run = True


In [ ]:
if run:
    split_count=5
    repeats = 10
    #skf = StratifiedKFold(n_splits=split_count,shuffle=True,random_state=15)
    #spl = skf.get_n_splits()
    rskf = RepeatedStratifiedKFold(n_splits=split_count,n_repeats=repeats,random_state=15)
    spl = rskf.get_n_splits()
    for feat in logreg_models:
        for (label,time_list) in [('day',day_list),('week',week_shift)]:
            for suffix in ['models','auc','aucpr','prec','rec']:
                keyval = label+'_'+suffix
                logreg_models[feat][keyval]=dict.fromkeys(time_list)
                for delta in day_list:
                    logreg_models[feat][keyval][delta]=dict.fromkeys(range(spl))

    # Loop over all the day deltas
    data_store={}
    for delta in day_list:
        target = 'lapse_bool_d'+str(delta)
        current_df = clean_feat.copy(deep=True).dropna(subset=[target]).reset_index(drop=True)
        Y = current_df[target]
        X = current_df.drop(misc_labels+day_test_labels+week_test_labels,axis=1)
        for i, (train, test) in enumerate(rskf.split(X,Y)):
            print("Split {}: Entry count: {}".format(i,len(test)))
            for feat in logreg_models:
                drop_list = logreg_models[feat]['drop']
                x_train = X.drop(drop_list,axis=1).loc[train].copy(deep=True)
                y_train = Y.loc[train].copy(deep=True)
                x_test = X.drop(drop_list,axis=1).loc[test].copy(deep=True)
                y_test = Y.loc[test].copy(deep=True)
                # Logistic regression
                model = sm.Logit(y_train, sm.add_constant(x_train))
                results = model.fit()
                yhat = model.predict(results.params,sm.add_constant(x_test))
                binary_pred = yhat
                # Area under ROC curve
                auc = roc_auc_score(y_test, yhat)
                # Precision measures ratio of true positives to (true positives + false positives)'
                # This is positive predictive value (fraction of predictions which are actually lapses)
                prec = precision_score(y_test,yhat>=0.5)
                # Recall measures ratio of true positive to (true positives + false negatives)
                # This is fraction of real lapses that were identified
                rec = recall_score(y_test,yhat>=0.5)
                # Add data to the corresponding dictionary
                logreg_models[feat]['day_models'][delta][i]=results
                logreg_models[feat]['day_auc'][delta][i]=auc
                logreg_models[feat]['day_prec'][delta][i]=prec
                logreg_models[feat]['day_rec'][delta][i]=rec

                # If featureset is 'ema_srm_lrm' and delta is 0 record precision recall information
                if feat=='ema_srm_lrm' and delta==0 and i==0:
                    data_store['train']=train
                    data_store['test']=test
                    data_store['x_train']=x_train
                    data_store['x_test']=x_test
                    data_store['y_train']=y_train
                    data_store['y_test']=y_test
                    data_store['model']=model
                    data_store['results']=results
                    data_store['yhat']=yhat
                    data_store['id_type_train']=current_df.loc[train]['id_type']
                    data_store['id_type_test']=current_df.loc[test]['id_type']

    # Assemble metric dataframe
    day_metric_list=[]
    for feat in logreg_models:
        for day in day_list:
            for i in range(split_count):
                auc = logreg_models[feat]['day_auc'][day][i]
                prec = logreg_models[feat]['day_prec'][day][i]
                rec = logreg_models[feat]['day_rec'][day][i]
                temp_df = pd.DataFrame({'features':feat,'target_day':day,'fold':i,'auc':auc,'precision':prec,'recall':rec},index=[0])
                day_metric_list.append(temp_df)
    day_metric_df = pd.concat(day_metric_list,ignore_index=True)
    day_metric_df.head()

In [ ]:
day_metric_df.head()

In [ ]:
fig,ax = plt.subplots(figsize=(20,15))
ax=sns.boxplot(data=day_metric_df.query("features!='limited_ema_only'"),x='features',y='auc',hue='target_day',palette='viridis_r')
ax.set_ylabel("AUC")
ax.set_xlabel("Feature Set")
#ax.set_yticks(range(0,30,2))
ax.set_title("Logistic regression AUCs for day-level predictions with varying feature sets, {}-fold, {} repeats".format(split_count,repeats))
#if remake_plots:
fig.savefig(os.path.join(plot_dir,'day_pred_auc_recent_lapse.pdf'),bbox_inches='tight',facecolor='w')
None

In [ ]:
fig,ax = plt.subplots(figsize=(20,15))
ax=sns.boxplot(data=day_metric_df.query("features!='limited_ema_only'"),x='features',y='aucpr',hue='target_day',palette='viridis_r')
ax.set_ylabel("AUCPR")
ax.set_xlabel("Feature Set")
#ax.set_yticks(range(0,30,2))
ax.set_title("Logistic regression AUCPRs for day-level predictions with varying feature sets, {}-fold, {} repeats".format(split_count,repeats))
#if remake_plots:
fig.savefig(os.path.join(plot_dir,'day_pred_aucpr_recent_lapse.pdf'),bbox_inches='tight',facecolor='w')
None

In [ ]:
logreg_models['ema_srm_lrm_minus']['day_models'][0][0].summary()

In [ ]:
# Loop over all the week deltas
if run:
    for delta in week_shift:
        target = 'lapse_bool_w'+str(delta)
        current_df = clean_feat.copy(deep=True).dropna(subset=[target])
        Y = current_df[target].reset_index(drop=True)
        X = current_df.drop(misc_labels+day_test_labels+week_test_labels,axis=1).reset_index(drop=True)
        for i, (train, test) in enumerate(rskf.split(X,Y)):
            print("Split {}: Entry count: {}".format(i,len(test)))
            for feat in logreg_models:
                drop_list = logreg_models[feat]['drop']
                x_train = X.drop(drop_list,axis=1).loc[train].copy(deep=True)
                y_train = Y.loc[train].copy(deep=True)
                x_test = X.drop(drop_list,axis=1).loc[test].copy(deep=True)
                y_test = Y.loc[test].copy(deep=True)
                # Logistic regression
                model = sm.Logit(y_train, sm.add_constant(x_train))
                results = model.fit()
                yhat = model.predict(results.params,sm.add_constant(x_test))
                binary_pred = yhat
                # Area under ROC curve
                auc = roc_auc_score(y_test, yhat)
                aucpr = average_precision_score(y_test,yhat)
                # Precision measures ratio of true positives to (true positives + false positives)'
                # This is positive predictive value (fraction of predictions which are actually lapses)
                prec = precision_score(y_test,yhat>=0.5)
                # Recall measures ratio of true positive to (true positives + false negatives)
                # This is fraction of real lapses that were identified
                rec = recall_score(y_test,yhat>=0.5)
                # Add data to the corresponding dictionary
                logreg_models[feat]['week_models'][delta][i]=results
                logreg_models[feat]['week_auc'][delta][i]=auc
                logreg_models[feat]['week_aucpr'][delta][i]=aucpr
                logreg_models[feat]['week_prec'][delta][i]=prec
                logreg_models[feat]['week_rec'][delta][i]=rec

    # Assemble metric dataframe
    week_metric_list=[]
    for feat in logreg_models:
        for shift in week_shift:
            for i in range(split_count):
                auc = logreg_models[feat]['week_auc'][shift][i]
                aucpr = logreg_models[feat]['week_aucpr'][shift][i]
                prec = logreg_models[feat]['week_prec'][shift][i]
                rec = logreg_models[feat]['week_rec'][shift][i]
                temp_df = pd.DataFrame({'features':feat,'week_shift':shift,'fold':i,'auc':auc,'aucpr':aucpr,'precision':prec,'recall':rec},index=[0])
                week_metric_list.append(temp_df)
    week_metric_df = pd.concat(week_metric_list,ignore_index=True)
    week_metric_df.head()

In [ ]:
fig,ax = plt.subplots(figsize=(20,15))
#cm = sns.cm.
ax=sns.boxplot(data=week_metric_df.query("features!='limited_ema_only'"),x='features',y='auc',hue='week_shift',palette='viridis_r')
ax.set_ylabel("AUC")
ax.set_xlabel("Feature Set")
#ax.set_yticks(range(0,30,2))
ax.set_title("Logistic regression AUCs for day-shifted, week-level predictions with varying feature sets, {}-fold, {} repeats".format(split_count,repeats))
#if remake_plots:
fig.savefig(os.path.join(plot_dir,'week_pred_auc_recent_lapse.pdf'),bbox_inches='tight',facecolor='w')
None

In [ ]:
fig,ax = plt.subplots(figsize=(20,15))
#cm = sns.cm.
ax=sns.boxplot(data=week_metric_df.query("features!='limited_ema_only'"),x='features',y='aucpr',hue='week_shift',palette='viridis_r')
ax.set_ylabel("AUCPR")
ax.set_xlabel("Feature Set")
#ax.set_yticks(range(0,30,2))
ax.set_title("Logistic regression AUCPRs for day-shifted, week-level predictions with varying feature sets, {}-fold, {} repeats".format(split_count,repeats))
#if remake_plots:
fig.savefig(os.path.join(plot_dir,'week_pred_aucpr_recent_lapse.pdf'),bbox_inches='tight',facecolor='w')

In [ ]:
logreg_models['ema']['week_models'][0][0].summary()

In [ ]:
# Run some tests on a single fold for visualization

# Get the data
target = 'lapse_bool_d0'
drop_list = logreg_models['ema_srm_lrm_minus']['drop']
current_df = clean_feat.copy(deep=True).dropna(subset=[target]).reset_index(drop=True)
Y = current_df[target]
X = current_df.drop(misc_labels+day_test_labels+week_test_labels+drop_list,axis=1)
x_train, x_test, y_train, y_test = train_test_split(X,Y,test_size=0.3,random_state=1,shuffle=True,stratify=Y)
test_idx=y_test.index.values
id_types = current_df.loc[test_idx]['id_type']
id0=id_types[id_types==0].index.values
id1=id_types[id_types==1].index.values
id2=id_types[id_types==2].index.values
id3=id_types[id_types==3].index.values
id4=id_types[id_types==4].index.values
# Define the grid of hyperparameters to search
hyperparameter_grid = {'n_estimators':randint(10,500),'max_features':['sqrt', None],
          'max_depth':randint(1,5),'min_samples_leaf':randint(1,10)}
          #'class_weight':['balanced']}
iter_count = 15

# Gradient boosting
GBmod = GradientBoostingClassifier(random_state=0)
# Set up the random search with 4-fold cross validation
clfGB = RandomizedSearchCV(GBmod,
            param_distributions=hyperparameter_grid,
            cv=4, n_iter=iter_count,
            scoring ='f1',
            n_jobs=-1)
clfGB.fit(x_train,y_train)

# Random forest
#Define the model
RFmod = RandomForestClassifier(random_state=0)

#Run the random search
clfRF = RandomizedSearchCV(RFmod,hyperparameter_grid,#model and parameters
                             cv=4,#number of cross validation folds
                             scoring='f1',#accuracy metric
                             n_iter=iter_count,
                             n_jobs=-1)#number of random parameter combinations
clfRF.fit(x_train,y_train)

# Sanity check using sklearn instead of statsmodels
model1 = LogisticRegression(penalty='none',max_iter=10000,fit_intercept=True)
clf1=model1.fit(x_train,y_train)
# Test with l2 regularization
model2 = LogisticRegression(penalty='l2',max_iter=10000,fit_intercept=True)
clf2=model2.fit(x_train,y_train)
# Test with class weighting and no regularization
model3 = LogisticRegression(penalty='none',max_iter=10000,fit_intercept=True,class_weight='balanced')
clf3=model3.fit(x_train,y_train)
# Test with class weighting and l1 regularization
model4 = LogisticRegression(penalty='l1',solver='liblinear',max_iter=10000,fit_intercept=True,class_weight='balanced')
clf4=model4.fit(x_train,y_train)

RocCurveDisplay.from_estimator(clf1,data_store['x_test'],data_store['y_test'])
PrecisionRecallDisplay.from_estimator(clf1,data_store['x_test'],data_store['y_test'])
RocCurveDisplay.from_estimator(clf2,data_store['x_test'],data_store['y_test'])
PrecisionRecallDisplay.from_estimator(clf2,data_store['x_test'],data_store['y_test'])
RocCurveDisplay.from_estimator(clf3,data_store['x_test'],data_store['y_test'])
PrecisionRecallDisplay.from_estimator(clf3,data_store['x_test'],data_store['y_test'])
RocCurveDisplay.from_estimator(clf4,data_store['x_test'],data_store['y_test'])
PrecisionRecallDisplay.from_estimator(clf4,data_store['x_test'],data_store['y_test'])
RocCurveDisplay.from_estimator(clfRF.best_estimator_,data_store['x_test'],data_store['y_test'])
PrecisionRecallDisplay.from_estimator(clfRF.best_estimator_,data_store['x_test'],data_store['y_test'])
RocCurveDisplay.from_estimator(clfGB.best_estimator_,data_store['x_test'],data_store['y_test'])
PrecisionRecallDisplay.from_estimator(clfGB.best_estimator_,data_store['x_test'],data_store['y_test'])

In [ ]:
plot_models = {'Logistic Regression':clf1,'Gradient Boosting Classifier':clfGB.best_estimator_}

for model in plot_models:
    classifier=plot_models[model]
    # Plot of AUC/PR
    fig,axs = plt.subplots(1,2,figsize=(20,10))
    #subfigs = fig.subfigures(1,,wspace=0.1)
    #leftaxs = subfigs[0].subplots(4,1,gridspec_kw={'height_ratios':[0.1,0.4,0.4,0.1]})
    #rightaxs = subfigs[1].subplots(4,1)

    # Plot the main ROC
    ax=axs[0]
    RocCurveDisplay.from_estimator(classifier,x_test,y_test,ax=ax,name="All test data")
    #RocCurveDisplay.from_estimator(clf1,x_test.loc[id0],y_test.loc[id0],ax=ax,name="Type 0")
    RocCurveDisplay.from_estimator(classifier,x_test.loc[id1],y_test.loc[id1],ax=ax,name="Type 1")
    RocCurveDisplay.from_estimator(classifier,x_test.loc[id2],y_test.loc[id2],ax=ax,name="Type 2")
    RocCurveDisplay.from_estimator(classifier,x_test.loc[id3],y_test.loc[id3],ax=ax,name="Type 3")
    RocCurveDisplay.from_estimator(classifier,x_test.loc[id4],y_test.loc[id4],ax=ax,name="Type 4")
    ax.set_title("ROC Curve")

    ax=axs[1]
    PrecisionRecallDisplay.from_estimator(classifier,x_test,y_test,ax=ax,name="All test data")
    #PrecisionRecallDisplay.from_estimator(clf1,x_test.loc[id0],y_test.loc[id0],ax=ax,name="Type 0")
    PrecisionRecallDisplay.from_estimator(classifier,x_test.loc[id1],y_test.loc[id1],ax=ax,name="Type 1")
    PrecisionRecallDisplay.from_estimator(classifier,x_test.loc[id2],y_test.loc[id2],ax=ax,name="Type 2")
    PrecisionRecallDisplay.from_estimator(classifier,x_test.loc[id3],y_test.loc[id3],ax=ax,name="Type 3")
    PrecisionRecallDisplay.from_estimator(classifier,x_test.loc[id4],y_test.loc[id4],ax=ax,name="Type 4")
    ax.set_title("ROC Curve")

    fig.suptitle("ROC/PR Curves for a sample data fold of same-day prediction - {}".format(model))
    fig.tight_layout()
    # Suppress output
    None

In [ ]:
data_store_list=[]
split_count=5
repeats = 10
rskf = RepeatedStratifiedKFold(n_splits=split_count,n_repeats=repeats,random_state=15)
spl = rskf.get_n_splits()
# Define the grid of hyperparameters to search
hyperparameter_grid = {'n_estimators':randint(10,500),'max_features':['sqrt', None],
          'max_depth':randint(1,5),'min_samples_leaf':randint(1,10)}
iter_count = 30
for delta in [0]:#[0,2,4,6,8]:
    target = 'lapse_bool_d'+str(delta)
    current_df = clean_feat.copy(deep=True).dropna(subset=[target]).reset_index(drop=True)
    Y = current_df[target]
    # Consider putting minus_labels back into the drop list
    X = current_df.drop(misc_labels+day_test_labels+week_test_labels,axis=1)
    for i, (train,test) in enumerate(rskf.split(X,Y)):
        print("Split {}".format(i))
        x_train = X.loc[train].copy(deep=True)
        y_train = Y.loc[train].copy(deep=True)
        x_test = X.loc[test].copy(deep=True)
        y_test = Y.loc[test].copy(deep=True)
        test_idx=y_test.index.values
        id_types = current_df.loc[test_idx]['id_type']
        id0=id_types[id_types==0].index.values
        id1=id_types[id_types==1].index.values
        id2=id_types[id_types==2].index.values
        id3=id_types[id_types==3].index.values
        id4=id_types[id_types==4].index.values
        # Logistic regression
        LRmodel = LogisticRegression(penalty=None,max_iter=10000,fit_intercept=True)
        LRclf=LRmodel.fit(x_train,y_train)
        yhat = LRmodel.predict_proba(x_test)[:,1]
        auc = roc_auc_score(y_test,yhat)
        aucpr = average_precision_score(y_test,yhat)
        aucs = {}
        aucprs = {}
        for (num,ids) in [('all',test_idx),('1',id1),('2',id2),('3',id3),('4',id4)]:
            yhat_i = LRmodel.predict_proba(x_test.loc[ids])[:,1]
            auc_i = roc_auc_score(y_test.loc[ids],yhat_i)
            aucpr_i = average_precision_score(y_test.loc[ids],yhat_i)
            aucs[num]=auc_i
            aucprs[num]=aucpr_i
            temp_df = pd.DataFrame({'auc':auc_i,'aucpr':aucpr_i,'id_type':num,
                                'model':"logistic regression",'delta':delta},index=[0])
            data_store_list.append(temp_df)
        # # Gradient boosting
        GBmodel = GradientBoostingClassifier()
        # Set up the random search with 4-fold cross validation
        clfGB = RandomizedSearchCV(GBmodel,
            param_distributions=hyperparameter_grid,
            cv=4, n_iter=iter_count,
            scoring ='f1',
            n_jobs=-1)
        clfGB.fit(x_train,y_train)
        yhat = clfGB.best_estimator_.predict_proba(x_test)[:,1]
        auc = roc_auc_score(y_test,yhat)
        aucpr = average_precision_score(y_test,yhat)
        aucs = {}
        aucprs = {}
        for (num,ids) in [('all',test_idx),('1',id1),('2',id2),('3',id3),('4',id4)]:
            yhat_i = clfGB.best_estimator_.predict_proba(x_test.loc[ids])[:,1]
            auc_i = roc_auc_score(y_test.loc[ids],yhat_i)
            aucpr_i = average_precision_score(y_test.loc[ids],yhat_i)
            aucs[num]=auc_i
            aucprs[num]=aucpr_i
            temp_df = pd.DataFrame({'auc':auc_i,'aucpr':aucpr_i,'id_type':num,
                                'model':"gradient boosting",'delta':delta},index=[0])
            data_store_list.append(temp_df)
        # temp_df = pd.DataFrame({'auc':auc,'auc1':aucs['1'],'auc2':aucs['2'],'auc3':aucs['3'],'auc4':aucs['4'],
        #                         'aucpr':aucpr,'aucpr1':aucprs['1'],'aucpr2':aucprs['2'],'aucpr3':aucprs['3'],'aucpr4':aucprs['4'],
        #                         'model':"gradient boosting",'delta':delta},index=[0])
            

data_store_df = pd.concat(data_store_list,ignore_index=True)
data_store_df.head()

In [ ]:
fig,axs = plt.subplots(1,2,figsize=(10,4))

ax=axs[0]
sns.boxplot(ax=ax,data=data_store_df.query("id_type=='all'"),x='delta',y='auc',hue='model')
ax.set_ylabel("AUC")
ax.set_xlabel("Day Delta")
ax.set_ylim([.5,1])

ax=axs[1]
sns.boxplot(ax=ax,data=data_store_df.query("id_type=='all'"),x='delta',y='aucpr',hue='model')
ax.set_ylabel("AUCPR")
ax.set_xlabel("Day Delta")
ax.set_ylim([0,.6])

fig.suptitle("AUC/AUCPR for LR/GB using full feature set, {}-fold, {} repeats".format(split_count,repeats))
fig.tight_layout()
#if remake_plots:
#fig.savefig(os.path.join(plot_dir,'full_feature_auc_aucpr.pdf'),bbox_inches='tight',facecolor='w')
None

In [ ]:
fig,axs = plt.subplots(1,2,figsize=(15,8))

ax=axs[0]
sns.boxplot(ax=ax,data=data_store_df.query("model=='logistic regression'"),x='delta',y='auc',hue='id_type')
ax.set_ylabel("AUC")
ax.set_xlabel("Day Delta")
ax.set_ylim([0,1])

ax=axs[1]
sns.boxplot(ax=ax,data=data_store_df.query("model=='logistic regression'"),x='delta',y='aucpr',hue='id_type')
ax.set_ylabel("AUCPR")
ax.set_xlabel("Day Delta")
ax.set_ylim([0,1])

fig.suptitle("AUC/AUCPR for LR using full feature set, {}-fold, {} repeats".format(split_count,repeats))
fig.tight_layout()
#if remake_plots:
#fig.savefig(os.path.join(plot_dir,'lr_full_feature_auc_aucpr.pdf'),bbox_inches='tight',facecolor='w')
None

In [ ]:
# Week level testing

week_store_list=[]
split_count=5
repeats = 10
rskf = RepeatedStratifiedKFold(n_splits=split_count,n_repeats=repeats,random_state=15)
spl = rskf.get_n_splits()
# Define the grid of hyperparameters to search
hyperparameter_grid = {'n_estimators':randint(10,500),'max_features':['sqrt', None],
          'max_depth':randint(1,5),'min_samples_leaf':randint(1,10)}
iter_count = 30
for delta in [0]:#[0,2,4,6,8]:
    target = 'lapse_bool_w'+str(delta)
    current_df = clean_feat.copy(deep=True).dropna(subset=[target]).reset_index(drop=True)
    Y = current_df[target]
    # Consider putting minus_labels back in the drop list
    X = current_df.drop(misc_labels+day_test_labels+week_test_labels,axis=1)
    for i, (train,test) in enumerate(rskf.split(X,Y)):
        print("Split {}".format(i))
        x_train = X.loc[train].copy(deep=True)
        y_train = Y.loc[train].copy(deep=True)
        x_test = X.loc[test].copy(deep=True)
        y_test = Y.loc[test].copy(deep=True)
        test_idx=y_test.index.values
        id_types = current_df.loc[test_idx]['id_type']
        id0=id_types[id_types==0].index.values
        id1=id_types[id_types==1].index.values
        id2=id_types[id_types==2].index.values
        id3=id_types[id_types==3].index.values
        id4=id_types[id_types==4].index.values
        # Logistic regression
        LRmodel = LogisticRegression(penalty=None,max_iter=10000,fit_intercept=True)
        LRclf=LRmodel.fit(x_train,y_train)
        yhat = LRmodel.predict_proba(x_test)[:,1]
        auc = roc_auc_score(y_test,yhat)
        aucpr = average_precision_score(y_test,yhat)
        aucs = {}
        aucprs = {}
        for (num,ids) in [('all',test_idx),('1',id1),('2',id2),('3',id3),('4',id4)]:
            yhat_i = LRmodel.predict_proba(x_test.loc[ids])[:,1]
            auc_i = roc_auc_score(y_test.loc[ids],yhat_i)
            aucpr_i = average_precision_score(y_test.loc[ids],yhat_i)
            aucs[num]=auc_i
            aucprs[num]=aucpr_i
            temp_df = pd.DataFrame({'auc':auc_i,'aucpr':aucpr_i,'id_type':num,
                                'model':"logistic regression",'delta':delta},index=[0])
            week_store_list.append(temp_df)
        # # Gradient boosting
        GBmodel = GradientBoostingClassifier()
        # Set up the random search with 4-fold cross validation
        clfGB = RandomizedSearchCV(GBmodel,
            param_distributions=hyperparameter_grid,
            cv=4, n_iter=iter_count,
            scoring ='f1',
            n_jobs=-1)
        clfGB.fit(x_train,y_train)
        yhat = clfGB.best_estimator_.predict_proba(x_test)[:,1]
        auc = roc_auc_score(y_test,yhat)
        aucpr = average_precision_score(y_test,yhat)
        aucs = {}
        aucprs = {}
        for (num,ids) in [('all',test_idx),('1',id1),('2',id2),('3',id3),('4',id4)]:
            yhat_i = clfGB.best_estimator_.predict_proba(x_test.loc[ids])[:,1]
            auc_i = roc_auc_score(y_test.loc[ids],yhat_i)
            aucpr_i = average_precision_score(y_test.loc[ids],yhat_i)
            aucs[num]=auc_i
            aucprs[num]=aucpr_i
            temp_df = pd.DataFrame({'auc':auc_i,'aucpr':aucpr_i,'id_type':num,
                                'model':"gradient boosting",'delta':delta},index=[0])
            week_store_list.append(temp_df)
        # temp_df = pd.DataFrame({'auc':auc,'auc1':aucs['1'],'auc2':aucs['2'],'auc3':aucs['3'],'auc4':aucs['4'],
        #                         'aucpr':aucpr,'aucpr1':aucprs['1'],'aucpr2':aucprs['2'],'aucpr3':aucprs['3'],'aucpr4':aucprs['4'],
        #                         'model':"gradient boosting",'delta':delta},index=[0])
            

week_store_df = pd.concat(week_store_list,ignore_index=True)
week_store_df.head()

In [ ]:
fig,axs = plt.subplots(1,2,figsize=(15,8))

ax=axs[0]
sns.boxplot(ax=ax,data=data_store_df.query("model=='gradient boosting'"),x='delta',y='auc',hue='id_type')
ax.set_ylabel("AUC")
ax.set_xlabel("Day Delta")
ax.set_ylim([0,1])

ax=axs[1]
sns.boxplot(ax=ax,data=data_store_df.query("model=='gradient boosting'"),x='delta',y='aucpr',hue='id_type')
ax.set_ylabel("AUCPR")
ax.set_xlabel("Day Delta")
ax.set_ylim([0,1])

fig.suptitle("AUC/AUCPR for GB using full feature set, {}-fold, {} repeats".format(split_count,repeats))
fig.tight_layout()
#if remake_plots:
#fig.savefig(os.path.join(plot_dir,'gb_full_feature_auc_aucpr.pdf'),bbox_inches='tight',facecolor='w')
None

In [ ]:
data_store_df['prediction']='same-day'
data_store_df.head()

In [ ]:
week_store_df['prediction']='within-week'
week_store_df.head()

In [ ]:
ref_df = pd.concat([data_store_df,week_store_df],ignore_index=True)

In [ ]:
fig,axs = plt.subplots(1,2,figsize=(10,4))

ax=axs[0]
sns.boxplot(ax=ax,data=ref_df.query("id_type=='all'"),x='prediction',y='auc',hue='model')
ax.set_ylabel("AUC")
ax.set_xlabel("Prediction")
ax.set_ylim([.5,1])

ax=axs[1]
sns.boxplot(ax=ax,data=ref_df.query("id_type=='all'"),x='prediction',y='aucpr',hue='model')
ax.set_ylabel("AUCPR")
ax.set_xlabel("Prediction")
ax.set_ylim([0,1])

fig.suptitle("AUC/AUCPR for LR/GB using full feature set, {}-fold, {} repeats, with recent lapse information".format(split_count,repeats))
fig.tight_layout()
#if remake_plots:
fig.savefig(os.path.join(plot_dir,'full_feature_auc_aucpr_day_week_ref_with_recent_lapse.pdf'),bbox_inches='tight',facecolor='w')
None
print(np.median(ref_df.query("id_type=='all'&model=='gradient boosting'&prediction=='within-week'").aucpr.to_numpy()))

In [ ]:
# Minus label check
data_store_list=[]
split_count=4
repeats = 10
rskf = RepeatedStratifiedKFold(n_splits=split_count,n_repeats=repeats,random_state=10)
spl = rskf.get_n_splits()
# Define the grid of hyperparameters to search
hyperparameter_grid = {'n_estimators':randint(10,500),'max_features':['sqrt', None],
          'max_depth':randint(1,5),'min_samples_leaf':randint(1,10)}
iter_count = 30
for delta in [0,4,8]:
    target = 'lapse_bool_d'+str(delta)
    current_df = clean_feat.copy(deep=True).dropna(subset=[target]).reset_index(drop=True)
    Y = current_df[target]
    X = current_df.drop(misc_labels+day_test_labels+week_test_labels+minus_labels,axis=1)
    for i, (train,test) in enumerate(rskf.split(X,Y)):
        print("Split {}".format(i))
        x_train = X.loc[train].copy(deep=True)
        y_train = Y.loc[train].copy(deep=True)
        x_test = X.loc[test].copy(deep=True)
        y_test = Y.loc[test].copy(deep=True)
        test_idx=y_test.index.values
        id_types = current_df.loc[test_idx]['id_type']
        id0=id_types[id_types==0].index.values
        id1=id_types[id_types==1].index.values
        id2=id_types[id_types==2].index.values
        id3=id_types[id_types==3].index.values
        id4=id_types[id_types==4].index.values
        # Logistic regression
        LRmodel = LogisticRegression(penalty=None,max_iter=10000,fit_intercept=True)
        LRclf=LRmodel.fit(x_train,y_train)
        yhat = LRmodel.predict_proba(x_test)[:,1]
        auc = roc_auc_score(y_test,yhat)
        aucpr = average_precision_score(y_test,yhat)
        aucs = {}
        aucprs = {}
        for (num,ids) in [('all',test_idx),('1',id1),('2',id2),('3',id3),('4',id4)]:
            yhat_i = LRmodel.predict_proba(x_test.loc[ids])[:,1]
            auc_i = roc_auc_score(y_test.loc[ids],yhat_i)
            aucpr_i = average_precision_score(y_test.loc[ids],yhat_i)
            aucs[num]=auc_i
            aucprs[num]=aucpr_i
            temp_df = pd.DataFrame({'auc':auc_i,'aucpr':aucpr_i,'id_type':num,
                                'model':"logistic regression",'delta':delta},index=[0])
            data_store_list.append(temp_df)
        # # Gradient boosting
        GBmodel = GradientBoostingClassifier()
        # Set up the random search with 4-fold cross validation
        clfGB = RandomizedSearchCV(GBmodel,
            param_distributions=hyperparameter_grid,
            cv=4, n_iter=iter_count,
            scoring ='f1',
            n_jobs=-1)
        clfGB.fit(x_train,y_train)
        yhat = clfGB.best_estimator_.predict_proba(x_test)[:,1]
        auc = roc_auc_score(y_test,yhat)
        aucpr = average_precision_score(y_test,yhat)
        aucs = {}
        aucprs = {}
        for (num,ids) in [('all',test_idx),('1',id1),('2',id2),('3',id3),('4',id4)]:
            yhat_i = clfGB.best_estimator_.predict_proba(x_test.loc[ids])[:,1]
            auc_i = roc_auc_score(y_test.loc[ids],yhat_i)
            aucpr_i = average_precision_score(y_test.loc[ids],yhat_i)
            aucs[num]=auc_i
            aucprs[num]=aucpr_i
            temp_df = pd.DataFrame({'auc':auc_i,'aucpr':aucpr_i,'id_type':num,
                                'model':"gradient boosting",'delta':delta,'featureset':'base'},index=[0])
            data_store_list.append(temp_df)
        # temp_df = pd.DataFrame({'auc':auc,'auc1':aucs['1'],'auc2':aucs['2'],'auc3':aucs['3'],'auc4':aucs['4'],
        #                         'aucpr':aucpr,'aucpr1':aucprs['1'],'aucpr2':aucprs['2'],'aucpr3':aucprs['3'],'aucpr4':aucprs['4'],
        #                         'model':"gradient boosting",'delta':delta},index=[0])
            

data_store_df_base = pd.concat(data_store_list,ignore_index=True)
data_store_df_base.to_csv('base_output.csv')
data_store_df_base.head()

In [ ]:
# Minus label check
data_store_list=[]
split_count=4
repeats = 10
rskf = RepeatedStratifiedKFold(n_splits=split_count,n_repeats=repeats,random_state=10)
spl = rskf.get_n_splits()
# Define the grid of hyperparameters to search
hyperparameter_grid = {'n_estimators':randint(10,500),'max_features':['sqrt', None],
          'max_depth':randint(1,5),'min_samples_leaf':randint(1,10)}
iter_count = 30
for delta in [0,4,8]:
    target = 'lapse_bool_d'+str(delta)
    current_df = clean_feat.copy(deep=True).dropna(subset=[target]).reset_index(drop=True)
    Y = current_df[target]
    X = current_df.drop(misc_labels+day_test_labels+week_test_labels,axis=1)
    for i, (train,test) in enumerate(rskf.split(X,Y)):
        print("Split {}".format(i))
        x_train = X.loc[train].copy(deep=True)
        y_train = Y.loc[train].copy(deep=True)
        x_test = X.loc[test].copy(deep=True)
        y_test = Y.loc[test].copy(deep=True)
        test_idx=y_test.index.values
        id_types = current_df.loc[test_idx]['id_type']
        id0=id_types[id_types==0].index.values
        id1=id_types[id_types==1].index.values
        id2=id_types[id_types==2].index.values
        id3=id_types[id_types==3].index.values
        id4=id_types[id_types==4].index.values
        # Logistic regression
        LRmodel = LogisticRegression(penalty=None,max_iter=10000,fit_intercept=True)
        LRclf=LRmodel.fit(x_train,y_train)
        yhat = LRmodel.predict_proba(x_test)[:,1]
        auc = roc_auc_score(y_test,yhat)
        aucpr = average_precision_score(y_test,yhat)
        aucs = {}
        aucprs = {}
        for (num,ids) in [('all',test_idx),('1',id1),('2',id2),('3',id3),('4',id4)]:
            yhat_i = LRmodel.predict_proba(x_test.loc[ids])[:,1]
            auc_i = roc_auc_score(y_test.loc[ids],yhat_i)
            aucpr_i = average_precision_score(y_test.loc[ids],yhat_i)
            aucs[num]=auc_i
            aucprs[num]=aucpr_i
            temp_df = pd.DataFrame({'auc':auc_i,'aucpr':aucpr_i,'id_type':num,
                                'model':"logistic regression",'delta':delta,'features':'minus'},index=[0])
            data_store_list.append(temp_df)
        # # Gradient boosting
        GBmodel = GradientBoostingClassifier()
        # Set up the random search with 4-fold cross validation
        clfGB = RandomizedSearchCV(GBmodel,
            param_distributions=hyperparameter_grid,
            cv=4, n_iter=iter_count,
            scoring ='f1',
            n_jobs=-1)
        clfGB.fit(x_train,y_train)
        yhat = clfGB.best_estimator_.predict_proba(x_test)[:,1]
        auc = roc_auc_score(y_test,yhat)
        aucpr = average_precision_score(y_test,yhat)
        aucs = {}
        aucprs = {}
        for (num,ids) in [('all',test_idx),('1',id1),('2',id2),('3',id3),('4',id4)]:
            yhat_i = clfGB.best_estimator_.predict_proba(x_test.loc[ids])[:,1]
            auc_i = roc_auc_score(y_test.loc[ids],yhat_i)
            aucpr_i = average_precision_score(y_test.loc[ids],yhat_i)
            aucs[num]=auc_i
            aucprs[num]=aucpr_i
            temp_df = pd.DataFrame({'auc':auc_i,'aucpr':aucpr_i,'id_type':num,
                                'model':"gradient boosting",'delta':delta,'features':'minus'},index=[0])
            data_store_list.append(temp_df)
        # temp_df = pd.DataFrame({'auc':auc,'auc1':aucs['1'],'auc2':aucs['2'],'auc3':aucs['3'],'auc4':aucs['4'],
        #                         'aucpr':aucpr,'aucpr1':aucprs['1'],'aucpr2':aucprs['2'],'aucpr3':aucprs['3'],'aucpr4':aucprs['4'],
        #                         'model':"gradient boosting",'delta':delta},index=[0])
            

data_store_df_minus4 = pd.concat(data_store_list,ignore_index=True)
data_store_df_minus4.to_csv('minus_output_4fold.csv')
data_store_df_minus4.head()

In [ ]:
print(data_store_df_base.shape)
data_store_df_minus4.shape

In [ ]:
test = pd.concat([data_store_df_base,data_store_df_minus4],ignore_index=True)
test.head()

In [ ]:
test['model_type']=test.apply(lambda row: str(row.model) +' '+str(row.features) if str(row.features)=='base' else str(row.model) +' recent lapse',axis=1)
test.head()

In [ ]:
test['model_type'].unique()

In [ ]:
fig,axs = plt.subplots(1,2,figsize=(10,4))
ordering = ['logistic regression base','logistic regression recent lapse','gradient boosting base','gradient boosting recent lapse']
ax=axs[0]
sns.boxplot(ax=ax,data=test.query("id_type=='all'"),x='delta',y='auc',hue='model_type',hue_order=ordering)
ax.set_ylabel("AUC")
ax.set_xlabel("Day Delta")
ax.set_ylim([.5,1])

ax=axs[1]
sns.boxplot(ax=ax,data=test.query("id_type=='all'"),x='delta',y='aucpr',hue='model_type',hue_order=ordering)
ax.set_ylabel("AUCPR")
ax.set_xlabel("Day Delta")
ax.set_ylim([0,.5])

fig.suptitle("AUC/AUCPR for LR/GB using full feature set, {}-fold, {} repeats".format(split_count,repeats))
fig.tight_layout()
#if remake_plots:
fig.savefig(os.path.join(plot_dir,'full_feature_auc_aucpr_lapse_boolean.pdf'),bbox_inches='tight',facecolor='w')
None